# 01 — Ingesta y Limpieza de Datos

## Contexto de Negocio

Este notebook documenta la ingesta y preparación de los datos de reservas del **Schedule P del NAIC**, publicados por la Casualty Actuarial Society (CAS). Estos datos son declaraciones regulatorias reales de aseguradoras en EE.UU., utilizados como referencia en la industria para investigación en reservas de siniestros.

**Fuente**: CAS Loss Reserving Database — NAIC Schedule P  
**Periodo**: Años de accidente 1988–1997, con desarrollo observado hasta 2006  
**Líneas de negocio**: 6 (Auto Personal, Auto Comercial, Workers' Comp, Malpráctica Médica, Otra Responsabilidad, Responsabilidad de Producto)  

### ¿Qué es el Schedule P?

El Schedule P es una sección de la declaración anual que las aseguradoras de P&C presentan ante el NAIC (National Association of Insurance Commissioners). Contiene triángulos de desarrollo de siniestros: pérdidas incurridas y pagadas por año de accidente y año de desarrollo. Es la base para estimar reservas IBNR (Incurred But Not Reported).

In [ ]:
import pandas as pd
import numpy as np
import os

RAW_DIR = '../data/raw'
PROCESSED_DIR = '../data/processed'

## 1. Carga de Datos Crudos

Cada archivo CSV corresponde a una línea de negocio (Part B, C, D, etc. del Schedule P). Las columnas clave son:

| Columna | Descripción |
|---------|-------------|
| `GRCODE` | Código del grupo asegurador |
| `GRNAME` | Nombre del grupo |
| `AccidentYear` | Año en que ocurrió el siniestro |
| `DevelopmentYear` | Año de evaluación |
| `DevelopmentLag` | Años transcurridos desde el accidente |
| `IncurLoss_*` | Pérdidas incurridas acumuladas (caso + IBNR) |
| `CumPaidLoss_*` | Pérdidas pagadas acumuladas |
| `EarnedPremDIR_*` | Prima ganada directa |

In [ ]:
LOB_MAP = {
    'ppauto_pos.csv': 'Private Passenger Auto',
    'comauto_pos.csv': 'Commercial Auto',
    'wkcomp_pos.csv': 'Workers Compensation',
    'medmal_pos.csv': 'Medical Malpractice',
    'othliab_pos.csv': 'Other Liability',
    'prodliab_pos.csv': 'Product Liability',
}

frames = []
for filename, lob in LOB_MAP.items():
    path = os.path.join(RAW_DIR, filename)
    df = pd.read_csv(path)
    
    # Detect and strip column suffix (each file has unique suffix: _B, _C, _D, etc.)
    incur_cols = [c for c in df.columns if c.startswith('IncurLoss')]
    if incur_cols:
        suffix = incur_cols[0].replace('IncurLoss', '')
        if suffix:
            renames = {c: c[:-len(suffix)] for c in df.columns if c.endswith(suffix)}
            df.rename(columns=renames, inplace=True)
    
    df['line_of_business'] = lob
    frames.append(df)
    print(f'{lob}: {len(df):,} rows, {df["GRCODE"].nunique()} companies')

raw = pd.concat(frames, ignore_index=True)
print(f'\nTotal: {len(raw):,} rows')

## 2. Exploración Inicial

Antes de filtrar, veamos la estructura general de los datos.

In [ ]:
print('Columnas principales:', [c for c in raw.columns if not c.endswith(('_C','_D','_F2','_h1','_R1'))])
print(f'\nAños de accidente: {raw["AccidentYear"].min()}–{raw["AccidentYear"].max()}')
print(f'Años de desarrollo: {raw["DevelopmentYear"].min()}–{raw["DevelopmentYear"].max()}')
print(f'Lags de desarrollo: {sorted(raw["DevelopmentLag"].unique())}')
print(f'\nCompañías por LOB:')
for lob in LOB_MAP.values():
    n = raw[raw['line_of_business']==lob]['GRCODE'].nunique()
    print(f'  {lob}: {n}')

In [ ]:
# Summary statistics for key financial columns
financial_cols = ['IncurLoss', 'CumPaidLoss', 'EarnedPremDIR', 'EarnedPremNet']
raw[financial_cols].describe().round(0)

## 3. Datos Procesados

El pipeline de datos (`data-pipeline/02_clean_triangles.py`) ya realizó:
1. Selección de ~5 compañías representativas por LOB (por volumen de prima)
2. Cálculo de columnas derivadas: `loss_ratio`, `paid_to_incurred`

Veamos el resultado.

In [ ]:
triangles = pd.read_parquet(os.path.join(PROCESSED_DIR, 'triangles.parquet'))
print(f'Triángulos procesados: {len(triangles):,} rows')
print(f'LOBs: {triangles["line_of_business"].unique().tolist()}')
print(f'Compañías: {sorted(triangles["GRCODE"].unique())}')
print(f'\nColumnas: {triangles.columns.tolist()}')
triangles.head()

## 4. Datos Sintéticos de Siniestros

El pipeline también generó ~50K siniestros individuales sintéticos calibrados a los agregados del CAS.

**Distribuciones actuariales utilizadas:**
- **Severidad**: Lognormal (estándar en la industria para montos de siniestros)
- **Frecuencia**: Poisson
- **Retraso de reporte**: Exponencial desplazada
- **Tiempo de liquidación**: Gamma

In [ ]:
claims = pd.read_parquet(os.path.join(PROCESSED_DIR, 'claims_synthetic.parquet'))
print(f'Siniestros sintéticos: {len(claims):,}')
print(f'\nColumnas: {claims.columns.tolist()}')
print(f'\nEstatus: {claims["claim_status"].value_counts().to_dict()}')
print(f'Bandas de severidad: {claims["severity_band"].value_counts().to_dict()}')
claims.head()

## 5. Resultados IBNR

Los resultados de reservas IBNR calculados con Chain-Ladder y Bornhuetter-Ferguson.

In [ ]:
ibnr = pd.read_parquet(os.path.join(PROCESSED_DIR, 'ibnr_results.parquet'))
print(f'Resultados IBNR: {len(ibnr)} rows')
print(f'\nColumnas: {ibnr.columns.tolist()}')
ibnr.head()

In [ ]:
lob_summary = pd.read_parquet(os.path.join(PROCESSED_DIR, 'lob_summary.parquet'))
lob_summary

## So What?

- Los datos del CAS Schedule P proporcionan triángulos de desarrollo reales de aseguradoras reguladas en EE.UU.
- Las 6 líneas de negocio cubren desde auto personal (desarrollo corto, alta frecuencia) hasta malpráctica médica (desarrollo largo, baja frecuencia/alta severidad).
- Los datos sintéticos de siniestros complementan los triángulos agregados con granularidad a nivel de siniestro individual.
- Los notebooks siguientes explorarán: distribuciones de severidad, descomposición frecuencia-severidad, triángulos de desarrollo, y ratios combinados.